### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___
Welcome to day 1 of the 5-Day Data Challenge! Today, we're going to be looking at how to deal with missing values. To get started, click the blue "Fork Notebook" button in the upper, right hand corner. This will create a private copy of this notebook that you can edit and play with. Once you're finished with the exercises, you can choose to make your notebook public to share with others. :)

> **Your turn!** As we work through this notebook, you'll see some notebook cells (a block of either code or text) that has "Your Turn!" written in it. These are exercises for you to do to help cement your understanding of the concepts we're talking about. Once you've written the code to answer a specific question, you can run the code by clicking inside the cell (box with code in it) with the code you want to run and then hit CTRL + ENTER (CMD + ENTER on a Mac). You can also click in a cell and then click on the right "play" arrow to the left of the code. If you want to run all the code in your notebook, you can use the double, "fast forward" arrows at the bottom of the notebook editor.

Here's what we're going to do today:

* [Take a first look at the data](#Take-a-first-look-at-the-data)
* [See how many missing data points we have](#See-how-many-missing-data-points-we-have)
* [Figure out why the data is missing](#Figure-out-why-the-data-is-missing)
* [Drop missing values](#Drop-missing-values)
* [Filling in missing values](#Filling-in-missing-values)

Let's get started!

# Quan sát dữ liệu đầu tiên

---

Điều đầu tiên chúng ta cần làm là tải các thư viện và bộ dữ liệu sẽ sử dụng. Hôm nay, tôi sẽ sử dụng một bộ dữ liệu về các diễn biến trong các trận đấu bóng đá Mỹ (American Football) để làm mẫu, còn bạn sẽ sử dụng bộ dữ liệu về các giấy phép xây dựng được cấp ở San Francisco.

> **Quan trọng!** Đảm bảo rằng bạn tự chạy ô mã (cell) này, nếu không phần mã còn lại của bạn sẽ không hoạt động!

In [1]:
# modules we'll use
import pandas as pd
import numpy as np

# read in all our data
nfl_data = pd.read_csv("../input/nflplaybyplay2009to2016/NFL Play by Play 2009-2017 (v4).csv")
sf_permits = pd.read_csv("../input/building-permit-applications-data/Building_Permits.csv")

# set seed for reproducibility
np.random.seed(0) 

/opt/conda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2698: DtypeWarning: Columns (25,51) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
/opt/conda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2698: DtypeWarning: Columns (22,32) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


The first thing I do when I get a new dataset is take a look at some of it. This lets me see that it all read in correctly and get an idea of what's going on with the data. In this case, I'm looking to see if I see any missing values, which will be reprsented with `NaN` or `None`.

In [2]:
# look at a few rows of the nfl_data file. I can see a handful of missing data already!
nfl_data.sample(5)

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
244485,2014-10-26,2014102607,18,3,1.0,00:39,1,939.0,12.0,TB,...,1.240299,0.225647,0.774353,0.245582,0.754418,0.225647,0.019935,-0.018156,0.038091,2014
115340,2011-11-20,2011112000,22,4,1.0,06:47,7,407.0,44.0,OAK,...,NaN,0.056036,0.943964,0.042963,0.957037,0.943964,0.013073,NaN,NaN,2011
68357,2010-11-14,2010111401,8,2,NaN,00:23,1,1823.0,0.0,CLE,...,NaN,0.365307,0.634693,0.384697,0.615303,0.634693,-0.019390,NaN,NaN,2010
368377,2017-09-24,2017092405,24,4,1.0,08:48,9,528.0,8.0,CLE,...,1.075660,0.935995,0.064005,0.921231,0.078769,0.064005,0.014764,0.003866,0.010899,2017
384684,2017-11-05,2017110505,11,2,1.0,09:15,10,2355.0,0.0,DEN,...,NaN,0.928474,0.071526,0.934641,0.065359,0.071526,-0.006166,NaN,NaN,2017


Yep, it looks like there's some missing values. What about in the sf_permits dataset?

In [5]:
# your turn! Look at a couple of rows from the sf_permits dataset. Do you notice any missing data?

# your code goes here :)
sf_permits.sample(5)

,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID
78651,201503120624,8,otc alterations permit,03/12/2015,4273,029,2986,NaN,26th,St,...,5.0,wood frame (5),5.0,wood frame (5),NaN,9.0,Mission,94110.0,"(37.74972976527956, -122.40963202424835)",1374057173006
177779,201707071220,8,otc alterations permit,07/07/2017,0453,004A,950,NaN,Bay,St,...,5.0,wood frame (5),5.0,wood frame (5),NaN,2.0,Russian Hill,94109.0,"(37.80472551510833, -122.42280760820965)",1469546420378
64978,M529127,8,otc alterations permit,10/21/2014,0478,011,1290,NaN,Chestnut,St,...,NaN,NaN,NaN,NaN,NaN,2.0,Russian Hill,94109.0,"(37.80244997614665, -122.42443255018165)",1359656465959
165958,201704073507,8,otc alterations permit,04/07/2017,6507,003A,1134,NaN,Noe,St,...,5.0,wood frame (5),5.0,wood frame (5),NaN,8.0,Noe Valley,94114.0,"(37.750866393629146, -122.43209633372915)",1458922436280
147902,201701237639,8,otc alterations permit,01/23/2017,0623,001,1755,NaN,Van Ness,Av,...,5.0,wood frame (5),5.0,wood frame (5),NaN,2.0,Pacific Heights,94109.0,"(37.791925784456105, -122.42306858292103)",1450933235988


# See how many missing data points we have
___

Ok, now we know that we do have some missing values. Let's see how many we have in each column. 

In [6]:
# get the number of missing data points per column
missing_values_count = nfl_data.isnull().sum()

# look at the # of missing points in the first ten columns
missing_values_count[0:10]

Date                0
GameID              0
Drive               0
qtr                 0
down            61154
time              224
TimeUnder           0
TimeSecs          224
PlayTimeDiff      444
SideofField       528
dtype: int64

That seems like a lot! It might be helpful to see what percentage of the values in our dataset were missing to give us a better sense of the scale of this problem:

In [7]:
# how many total missing values do we have?
total_cells = np.product(nfl_data.shape)
total_missing = missing_values_count.sum()

# percent of data that is missing
(total_missing/total_cells) * 100

24.87214126835169

Wow, almost a quarter of the cells in this dataset are empty! In the next step, we're going to take a closer look at some of the columns with missing values and try to figure out what might be going on with them.

In [8]:
# your turn! Find out what percent of the sf_permits dataset is missing
missing_value_cnt = sf_permits.isnull().sum()
missing_value_cnt[0:20]

Permit Number                            0
Permit Type                              0
Permit Type Definition                   0
Permit Creation Date                     0
Block                                    0
Lot                                      0
Street Number                            0
Street Number Suffix                196684
Street Name                              0
Street Suffix                         2768
Unit                                169421
Unit Suffix                         196939
Description                            290
Current Status                           0
Current Status Date                      0
Filed Date                               0
Issued Date                          14940
Completed Date                      101709
First Construction Document Date     14946
Structural Notification             191978
dtype: int64

In [12]:
total_cell = np.product(sf_permits.shape)
total_miss = missing_value_cnt.sum()
print (total_miss/total_cell * 100)


26.26002315058403


# Tìm hiểu lý do tại sao dữ liệu bị thiếu

---

Đây là lúc chúng ta bước vào một phần của khoa học dữ liệu mà tôi thích gọi là "trực giác dữ liệu" (data intuition) — nghĩa là "thực sự quan sát dữ liệu và cố gắng tìm hiểu lý do tại sao nó lại như vậy, cũng như điều đó sẽ ảnh hưởng thế nào đến phân tích của bạn". Đây có thể là một phần gây nản lòng trong khoa học dữ liệu, đặc biệt nếu bạn là người mới trong ngành và chưa có nhiều kinh nghiệm. Để xử lý các giá trị bị thiếu, bạn sẽ cần dùng trực giác của mình để tìm ra nguyên nhân vì sao giá trị đó lại thiếu. Một trong những câu hỏi quan trọng nhất bạn có thể tự đặt ra để làm rõ điều này là:

> **Giá trị này bị thiếu là do nó không được ghi chép lại hay do nó không tồn tại?**

Nếu một giá trị bị thiếu vì nó không tồn tại (chẳng hạn như chiều cao của đứa con đầu lòng của một người chưa có con) thì việc cố gắng đoán xem nó là gì sẽ hoàn toàn vô nghĩa. Những giá trị này có lẽ bạn sẽ muốn giữ nguyên dưới dạng NaN. Mặt khác, nếu một giá trị bị thiếu vì nó chưa được ghi lại, bạn có thể thử đoán xem nó có thể là gì dựa trên các giá trị khác trong cùng cột và hàng đó. (Việc này được gọi là "quy gán" — imputation — và chúng ta sẽ học cách thực hiện ở phần tiếp theo! :)

Hãy cùng đi qua một ví dụ. Khi nhìn vào số lượng giá trị bị thiếu trong dataframe nfl_data, tôi nhận thấy rằng cột `TimesSec` có rất nhiều giá trị bị thiếu:

In [13]:
# look at the # of missing points in the first ten columns
missing_values_count[0:10]

Date                0
GameID              0
Drive               0
qtr                 0
down            61154
time              224
TimeUnder           0
TimeSecs          224
PlayTimeDiff      444
SideofField       528
dtype: int64

Bằng cách xem [tài liệu hướng dẫn](https://www.kaggle.com/maxhorowitz/nflplaybyplay2009to2016), tôi có thể thấy rằng cột này chứa thông tin về số giây còn lại của trận đấu khi lượt chơi diễn ra. Điều này có nghĩa là các giá trị này có khả năng bị thiếu do chúng không được ghi chép lại, chứ không phải vì chúng không tồn tại. Do đó, việc chúng ta cố gắng đoán xem chúng nên là giá trị gì sẽ hợp lý hơn là chỉ để nguyên dưới dạng NA.

Mặt khác, có những trường khác như `PenalizedTeam` cũng có rất nhiều ô dữ liệu bị thiếu. Tuy nhiên, trong trường hợp này, trường này bị thiếu là vì nếu không có lỗi phạt nào xảy ra thì việc nói đội *nào* bị phạt sẽ trở nên vô nghĩa. Đối với cột này, việc để trống hoặc thêm một giá trị thứ ba như "neither" (không đội nào) và dùng nó để thay thế cho các giá trị NA sẽ hợp lý hơn.

> **Mẹo:** Đây là thời điểm tuyệt vời để đọc kỹ tài liệu hướng dẫn của bộ dữ liệu nếu bạn chưa làm việc đó! Nếu bạn đang làm việc với một bộ dữ liệu nhận được từ người khác, bạn cũng có thể thử liên hệ với họ để biết thêm thông tin.

Nếu bạn đang thực hiện phân tích dữ liệu một cách rất cẩn thận, thì đây là lúc bạn sẽ xem xét riêng từng cột để tìm ra chiến lược tốt nhất cho việc lấp đầy các giá trị bị thiếu đó. Trong phần còn lại của notebook này, chúng ta sẽ tìm hiểu một số kỹ thuật "nhanh gọn" (quick and dirty) có thể giúp bạn xử lý các giá trị bị thiếu, nhưng có thể cũng sẽ làm mất đi một số thông tin hữu ích hoặc thêm nhiễu vào dữ liệu của bạn.

## Đến lượt bạn!

* Hãy nhìn vào các cột `Street Number Suffix` (Hậu tố số nhà) và `Zipcode` (Mã bưu chính) từ bộ dữ liệu `sf_permits`. Cả hai cột này đều chứa các giá trị bị thiếu. Cột nào (nếu có) bị thiếu do giá trị đó không tồn tại? Cột nào (nếu có) bị thiếu do không được ghi chép lại?

---

### 1. Cột `Street Number Suffix` (Hậu tố số nhà)

* **Nguyên nhân:** **Không tồn tại** (*doesn't exist*).
* **Giải thích:** Hậu tố số nhà thường là các chữ cái bổ sung đính kèm theo số nhà (ví dụ: nhà số `123A` hoặc `123B`, trong đó `A` và `B` là hậu tố). Đại đa số các địa chỉ nhà thông thường chỉ có số nhà đơn thuần (như `123 Main Street`) chứ không có hậu tố. Do đó, nếu ô này bị bỏ trống, điều đó có nghĩa là địa chỉ đó thực sự không có hậu tố.
* **Hướng xử lý:** Giữ nguyên giá trị `NaN` hoặc thay thế bằng chuỗi rỗng `""`.

---

### 2. Cột `Zipcode` (Mã bưu chính)

* **Nguyên nhân:** **Không được ghi chép lại** (*wasn't recorded*).
* **Giải thích:** Tất cả các công trình/địa chỉ nằm trong thành phố San Francisco chắc chắn đều thuộc về một mã Zipcode nhất định. Một địa chỉ không thể nào "không có mã Zipcode". Việc dữ liệu bị trống ở cột này chỉ có thể là do người nhập liệu quên điền, bỏ sót, hoặc hệ thống chưa ghi nhận thông tin đó lúc thu thập.
* **Hướng xử lý:** Có thể quy gán (impute) bằng cách suy ra từ các thông tin khác cùng hàng như tên đường (`Street Name`), số nhà (`Street Number`), hoặc vị trí tọa độ địa lý.

# Xóa các giá trị bị thiếu

---

Nếu bạn đang vội hoặc không có lý do để tìm hiểu nguyên nhân tại sao dữ liệu bị thiếu, một lựa chọn bạn có là xóa bỏ hoàn toàn bất kỳ hàng hoặc cột nào chứa giá trị bị thiếu. (Lưu ý: Tôi thường không khuyến khích phương pháp này đối với các dự án quan trọng! Việc bỏ thời gian để đi qua dữ liệu và thực sự kiểm tra từng cột chứa giá trị thiếu một cách kỹ lưỡng để hiểu rõ bộ dữ liệu của bạn thường sẽ xứng đáng hơn.)

Nếu bạn chắc chắn muốn xóa các hàng có giá trị bị thiếu, thư viện pandas có một hàm rất tiện lợi là `dropna()` để giúp bạn thực hiện việc này. Hãy cùng thử nghiệm hàm này trên bộ dữ liệu NFL của chúng ta!

---

In [14]:
# remove all the rows that contain a missing value
nfl_data.dropna()

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season


Trời ơi, có vẻ như việc đó đã xóa sạch toàn bộ dữ liệu của chúng ta rồi! 😱 Điều này xảy ra là do mỗi hàng trong bộ dữ liệu của chúng ta đều có ít nhất một giá trị bị thiếu. Có lẽ chúng ta sẽ may mắn hơn nếu xóa tất cả các *cột* có chứa ít nhất một giá trị bị thiếu thay vì xóa hàng.

In [15]:
# remove all columns with at least one missing value
columns_with_na_dropped = nfl_data.dropna(axis=1)
columns_with_na_dropped.head()

,Date,GameID,Drive,qtr,TimeUnder,ydstogo,ydsnet,PlayAttempted,Yards.Gained,sp,...,Timeout_Indicator,Timeout_Team,posteam_timeouts_pre,HomeTimeouts_Remaining_Pre,AwayTimeouts_Remaining_Pre,HomeTimeouts_Remaining_Post,AwayTimeouts_Remaining_Post,ExPoint_Prob,TwoPoint_Prob,Season
0,2009-09-10,2009091000,1,1,15,0,0,1,39,0,...,0,None,3,3,3,3,3,0.0,0.0,2009
1,2009-09-10,2009091000,1,1,15,10,5,1,5,0,...,0,None,3,3,3,3,3,0.0,0.0,2009
2,2009-09-10,2009091000,1,1,15,5,2,1,-3,0,...,0,None,3,3,3,3,3,0.0,0.0,2009
3,2009-09-10,2009091000,1,1,14,8,2,1,0,0,...,0,None,3,3,3,3,3,0.0,0.0,2009
4,2009-09-10,2009091000,1,1,14,8,2,1,0,0,...,0,None,3,3,3,3,3,0.0,0.0,2009


In [16]:
# just how much data did we lose?
print("Columns in original dataset: %d \n" % nfl_data.shape[1])
print("Columns with na's dropped: %d" % columns_with_na_dropped.shape[1])

Columns in original dataset: 102 

Columns with na's dropped: 41


Chúng ta đã làm mất khá nhiều dữ liệu, nhưng tại thời điểm này, chúng ta đã xóa thành công tất cả các giá trị `NaN` khỏi dữ liệu của mình.

In [18]:
# Your turn! Try removing all the rows from the sf_permits dataset that contain missing values. How many are left?
# remove all the rows that contain a missing value
sf_permits.dropna()

,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID


In [19]:
# Now try removing all the columns with empty values. Now how much of your data is left?
# remove all columns with at least one missing value
columns_with_na_dropped = sf_permits.dropna(axis=1)
columns_with_na_dropped.head()


,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Name,Current Status,Current Status Date,Filed Date,Record ID
0,201505065519,4,sign - erect,05/06/2015,0326,023,140,Ellis,expired,12/21/2017,05/06/2015,1380611233945
1,201604195146,4,sign - erect,04/19/2016,0306,007,440,Geary,issued,08/03/2017,04/19/2016,1420164406718
2,201605278609,3,additions alterations or repairs,05/27/2016,0595,203,1647,Pacific,withdrawn,09/26/2017,05/27/2016,1424856504716
3,201611072166,8,otc alterations permit,11/07/2016,0156,011,1230,Pacific,complete,07/24/2017,11/07/2016,1443574295566
4,201611283529,6,demolitions,11/28/2016,0342,001,950,Market,issued,12/01/2017,11/28/2016,144548169992


In [20]:
print("Columns in original dataset: %d \n" % sf_permits.shape[1])
print("Columns with na's dropped: %d" % columns_with_na_dropped.shape[1])

Columns in original dataset: 43 

Columns with na's dropped: 12


# Tự động điền các giá trị bị thiếu

---

Một lựa chọn khác là thử điền các giá trị bị thiếu. Ở phần tiếp theo này, tôi sẽ lấy một đoạn dữ liệu nhỏ từ bộ dữ liệu NFL để nó hiển thị được gọn đẹp hơn.

In [21]:
# get a small subset of the NFL dataset
subset_nfl_data = nfl_data.loc[:, 'EPA':'Season'].head()
subset_nfl_data

,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2.014474,NaN,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,-1.402760,NaN,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2.097796,NaN,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


Chúng ta có thể sử dụng hàm `fillna()` của Pandas để tự động điền các giá trị bị thiếu trong một dataframe. Một lựa chọn là chỉ định rõ giá trị mà chúng ta muốn dùng để thay thế cho các giá trị `NaN`. Ở đây, tôi đang yêu cầu thay thế toàn bộ các giá trị `NaN` bằng số 0.

In [22]:
# replace all NA's with 0
subset_nfl_data.fillna(0)

,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2.014474,0.000000,0.000000,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,0.000000,0.000000,2009
1,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,-1.402760,0.000000,0.000000,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,0.000000,0.000000,2009
3,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2.097796,0.000000,0.000000,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,0.000000,0.000000,2009


Tôi cũng có thể nhạy bén hơn một chút bằng cách thay thế các giá trị bị thiếu bằng chính giá trị xuất hiện ngay phía sau nó trong cùng một cột. (Cách làm này rất hợp lý đối với các bộ dữ liệu mà các quan sát có một thứ tự logic nhất định nào đó.)

In [23]:
# replace all NA's the value that comes directly after it in the same column, 
# then replace all the reamining na's with 0
subset_nfl_data.fillna(method = 'bfill', axis=0).fillna(0)

,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2.014474,-1.068169,1.146076,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,-0.032244,0.036899,2009
1,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,-1.402760,3.318841,-5.031425,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,0.106663,-0.156239,2009
3,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2.097796,0.000000,0.000000,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,0.000000,0.000000,2009



Việc điền các giá trị bị thiếu còn được gọi là "quy gán" (imputation), và bạn có thể tìm thêm các bài tập về chủ đề này [trong bài học này, cũng được đính kèm ở mục "Rèn luyện thêm!"](https://www.kaggle.com/dansbecker/handling-missing-values). Tuy nhiên, trước tiên, tại sao bạn không thử thay thế một số giá trị bị thiếu trong bộ dữ liệu `sf_permits` xem sao?

---

In [27]:
# Your turn! Try replacing all the NaN's in the sf_permits data with the one that
# comes directly after it and then replacing any remaining NaN's with 0

# Điền tất cả các ô trống bằng số 0
sf_permits.fillna(0).head()



,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID
0,201505065519,4,sign - erect,05/06/2015,0326,023,140,0,Ellis,St,...,3.0,constr type 3,0.0,0,0,3.0,Tenderloin,94102.0,"(37.785719256680785, -122.40852313194863)",1380611233945
1,201604195146,4,sign - erect,04/19/2016,0306,007,440,0,Geary,St,...,3.0,constr type 3,0.0,0,0,3.0,Tenderloin,94102.0,"(37.78733980600732, -122.41063199757738)",1420164406718
2,201605278609,3,additions alterations or repairs,05/27/2016,0595,203,1647,0,Pacific,Av,...,1.0,constr type 1,1.0,constr type 1,0,3.0,Russian Hill,94109.0,"(37.7946573324287, -122.42232562979227)",1424856504716
3,201611072166,8,otc alterations permit,11/07/2016,0156,011,1230,0,Pacific,Av,...,5.0,wood frame (5),5.0,wood frame (5),0,3.0,Nob Hill,94109.0,"(37.79595867909168, -122.41557405519474)",1443574295566
4,201611283529,6,demolitions,11/28/2016,0342,001,950,0,Market,St,...,3.0,constr type 3,0.0,0,0,6.0,Tenderloin,94102.0,"(37.78315261897309, -122.40950883997789)",144548169992


In [28]:
# Điền bằng giá trị phía trước (forward fill) hoặc phía sau (backward fill)
sf_permits.fillna(method='bfill').head()


,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID
0,201505065519,4,sign - erect,05/06/2015,0326,023,140,A,Ellis,St,...,3.0,constr type 3,1.0,constr type 1,Y,3.0,Tenderloin,94102.0,"(37.785719256680785, -122.40852313194863)",1380611233945
1,201604195146,4,sign - erect,04/19/2016,0306,007,440,A,Geary,St,...,3.0,constr type 3,1.0,constr type 1,Y,3.0,Tenderloin,94102.0,"(37.78733980600732, -122.41063199757738)",1420164406718
2,201605278609,3,additions alterations or repairs,05/27/2016,0595,203,1647,A,Pacific,Av,...,1.0,constr type 1,1.0,constr type 1,Y,3.0,Russian Hill,94109.0,"(37.7946573324287, -122.42232562979227)",1424856504716
3,201611072166,8,otc alterations permit,11/07/2016,0156,011,1230,A,Pacific,Av,...,5.0,wood frame (5),5.0,wood frame (5),Y,3.0,Nob Hill,94109.0,"(37.79595867909168, -122.41557405519474)",1443574295566
4,201611283529,6,demolitions,11/28/2016,0342,001,950,A,Market,St,...,3.0,constr type 3,1.0,constr type 1,Y,6.0,Tenderloin,94102.0,"(37.78315261897309, -122.40950883997789)",144548169992


Và đó là tất cả cho ngày hôm nay! Nếu bạn có bất kỳ câu hỏi nào, hãy nhớ đăng chúng ở phần bình luận bên dưới hoặc [trên diễn đàn](https://www.kaggle.com/questions-and-answers).

Hãy nhớ rằng notebook của bạn mặc định ở chế độ riêng tư (private), và để chia sẻ nó với người khác hoặc yêu cầu trợ giúp, bạn sẽ cần chuyển nó sang chế độ công khai (public). Đầu tiên, bạn cần lưu một phiên bản notebook hiển thị công việc hiện tại bằng cách nhấn vào nút "Commit & Run". (Công việc của bạn được tự động lưu lại, nhưng việc quản lý phiên bản cho phép bạn quay lại xem công việc trông như thế nào tại thời điểm bạn lưu. Nó cũng giúp bạn chia sẻ một notebook đã được biên dịch hoàn chỉnh thay vì chỉ là mã thô.) Sau đó, khi notebook chạy xong, bạn có thể chuyển đến thẻ Settings ở bảng điều khiển bên trái (bạn có thể phải mở rộng bảng này bằng cách nhấn nút [<] bên cạnh nút "Commit & Run") và đặt menu thả xuống "Visibility" thành "Public".

# Rèn luyện thêm!

---

Nếu bạn đang tìm kiếm thêm các bài tập thực hành xử lý giá trị bị thiếu, hãy tham khảo các bài tập cộng điểm* này:

* [Handling Missing Values](https://www.kaggle.com/dansbecker/handling-missing-values): Trong notebook này, Dan sẽ giới thiệu cho bạn một số phương pháp quy gán dữ liệu bị thiếu bằng công cụ imputer của scikit-learn.
* Nhìn lại cột `Zipcode` trong bộ dữ liệu `sf_permits`, cột này có một số giá trị bị thiếu. Bạn sẽ làm thế nào để tìm ra mã zipcode thực tế của từng địa chỉ? (Bạn có thể thử dùng một bộ dữ liệu khác. Bạn có thể tìm kiếm các bộ dữ liệu về San Francisco trên [Danh sách bộ dữ liệu](https://www.kaggle.com/datasets).)

* không có điểm thưởng thực tế nào được trao khi hoàn thành thử thách này đâu, bạn chỉ là học được cách làm sạch dữ liệu xịn hơn thôi :P